In [0]:
%sql
create catalog if not exists projects
comment "This is a catalog projects"

In [0]:
%sql
create schema if not exists projects.healthcare
comment "This is a schema healthcare project";

-- drop schema projects.heathcare CASCADE 

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS projects.healthcare.healthcareVolume
COMMENT "This is a volume for healthcare project";

-- drop volume projects.healthcare.volume 

In [0]:
# databricks %fs %cp "C:\Users\YourName\Documents\sales.csv" dbfs:/Volumes/main/default/my_uploads/sales.csv


In [0]:
import re;

# 1. Define your path and target table name
file_path = "/Volumes/projects/healthcare/healthcarevolume/healthcare_dataset.csv"
table_name = "projects.healthcare.bronze_patient"

# 2. Read the CSV into a Spark DataFrame
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_path)
)

# 3. Clean column names (replaces spaces and special characters with underscores)
clean_columns = [re.sub(r'[ ,;{}()\n\t=]+', '_', col.strip()) for col in df.columns]
df_clean = df.toDF(*clean_columns)

# # Another way of cleaning the column names - Clean column names
# for old_name in spark_df.columns:
#     new_name = re.sub(r'[^a-zA-Z0-9_]', '_', old_name)
 
#     # Remove consecutive underscores
#     new_name = re.sub(r'_+', '_', new_name)
 
#     # Remove leading/trailing underscores
#     new_name = new_name.strip('_')
 
#     spark_df = spark_df.withColumnRenamed(old_name, new_name)
 
# # Check updated column names
# print(spark_df.columns)

# 4. Write the DataFrame to a Delta Table
df_clean.write.mode("overwrite").saveAsTable(table_name)

In [0]:
%sql
select * from projects.healthcare.bronze_patient;

In [0]:
# Load the table into a DataFrame
df = spark.read.table("projects.healthcare.bronze_patient")

# Display the first 10 rows in Databricks
display(df)

In [0]:
df = spark.read.table("projects.healthcare.bronze_patient")
df.count()
df.distinct().count()
